# ¿Es significativa la pendiente? Regresión simple con SciPy, statsmodels y scikit-learn

En este notebook vamos a:

1. Ajustar una recta con **statsmodels** y leer su tabla de resultados: coeficiente, error estándar, estadístico t, p-valor e intervalo de confianza.
2. Entender qué es el error estándar de la pendiente y de qué depende.
3. Comprobar a mano de dónde sale el p-valor de la pendiente.
4. Ver un caso en el que la pendiente **no** es significativa y qué podemos concluir, y cómo un solo punto atípico puede cambiar la conclusión.
5. Simular qué ocurre cuando la pendiente real es 0 y cómo influye el tamaño de la muestra.
6. Ajustar la misma recta con NumPy, SciPy, statsmodels y **scikit-learn**, y comparar qué da cada una.

El contraste que usamos es el de la pendiente:

$$H_0: \beta_1 = 0 \qquad H_1: \beta_1 \neq 0, \qquad t = \frac{\hat{\beta}_1}{SE(\hat{\beta}_1)} \sim t_{n-2} \text{ si } H_0 \text{ es cierta.}$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

## 1. Los datos: publicidad y ventas

Diez semanas de una tienda: inversión en publicidad y ventas, las dos en miles de euros.

In [ ]:
publicidad = pd.DataFrame(
    {
        "publicidad": [10, 15, 12, 18, 20, 8, 14, 16, 11, 13],
        "ventas": [45, 60, 50, 70, 75, 40, 55, 65, 48, 52],
    }
)

plt.scatter(publicidad["publicidad"], publicidad["ventas"])
plt.xlabel("Publicidad (miles €)")
plt.ylabel("Ventas (miles €)")
plt.title("Publicidad y ventas")
plt.show()

## 2. Ajuste con statsmodels

Con la interfaz de fórmulas (`statsmodels.formula.api`) se escribe el modelo como `"Y ~ X"`. El intercepto se añade solo.

In [ ]:
modelo = smf.ols("ventas ~ publicidad", data=publicidad).fit()
print(modelo.summary())

### Cómo leer la tabla de coeficientes

La parte que nos interesa es la tabla central. Hay una fila por coeficiente (`Intercept` y `publicidad`):

- `coef`: la estimación, $\hat{\beta}_0$ o $\hat{\beta}_1$.
- `std err`: su error estándar, $SE(\hat{\beta})$: cuánto variaría la estimación con otra muestra.
- `t`: `coef / std err`.
- `P>|t|`: el p-valor del contraste $H_0: \beta = 0$. Un `0.000` significa p < 0.0005.
- `[0.025  0.975]`: el intervalo de confianza al 95 %.

La pregunta "¿es significativa la pendiente?" se responde en la fila `publicidad`: si el p-valor es menor que 0.05 (o, lo que es lo mismo, si el intervalo no contiene el 0), rechazamos $H_0$.

Las pruebas de la parte de abajo del `summary` (`Omnibus`, `Jarque-Bera`, `Durbin-Watson`) son sobre los residuos: con 10 datos dicen poco.

Cada número de la tabla también se puede sacar por separado:

In [ ]:
tabla = pd.DataFrame(
    {
        "coef": modelo.params,
        "std err": modelo.bse,
        "t": modelo.tvalues,
        "p-valor": modelo.pvalues,
        "IC 2.5 %": modelo.conf_int()[0],
        "IC 97.5 %": modelo.conf_int()[1],
    }
)
tabla.round(4)

## 3. El error estándar de la pendiente, $SE(\hat{\beta}_1)$

### La pendiente es una variable aleatoria

La pendiente 3.047 se ha calculado con **diez semanas concretas**. Si la tienda repitiera el experimento con otras diez semanas y la misma publicidad, las ventas no serían exactamente las mismas (hay ruido: el clima, la competencia, el azar), y la recta ajustada saldría algo distinta.

Pasa lo mismo con cualquier estadístico, como la media muestral: cada muestra da un valor distinto. El **error estándar de la pendiente**, $SE(\hat{\beta}_1)$, es la desviación típica de esas pendientes: cuánto se mueve la pendiente estimada de una muestra a otra.

Lo podemos ver con una simulación. Suponemos que la recta real es la que hemos estimado ($\beta_0 = 14.26$, $\beta_1 = 3.05$) y que el ruido tiene desviación típica $\sigma = 1.35$ (el error típico de la regresión, que calculamos más abajo). Generamos 5 000 muestras de diez semanas, con la misma publicidad, y ajustamos una recta a cada una.

In [ ]:
rng = np.random.default_rng(0)
x = publicidad["publicidad"].to_numpy()
beta0_real, beta1_real, sigma_real = 14.26, 3.05, 1.35

pendientes_sim = []
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i in range(5000):
    y_sim = beta0_real + beta1_real * x + rng.normal(0, sigma_real, len(x))
    res = stats.linregress(x, y_sim)
    pendientes_sim.append(res.slope)
    if i < 30:  # dibujamos las 30 primeras rectas
        xs = np.array([x.min(), x.max()])
        axes[0].plot(xs, res.intercept + res.slope * xs, color="C0", alpha=0.3)
axes[0].scatter(
    publicidad["publicidad"],
    publicidad["ventas"],
    color="C1",
    zorder=3,
    label="datos reales",
)
axes[0].set_xlabel("Publicidad (miles €)")
axes[0].set_ylabel("Ventas (miles €)")
axes[0].set_title("30 rectas ajustadas a 30 muestras simuladas")
axes[0].legend()

axes[1].hist(pendientes_sim, bins=50)
axes[1].set_xlabel("Pendiente estimada")
axes[1].set_title("Pendientes de 5 000 muestras")
plt.show()

print(f"Desviación típica de las 5 000 pendientes: {np.std(pendientes_sim):.3f}")
print(f"SE(b1) que da statsmodels con una sola muestra: {modelo.bse['publicidad']:.3f}")

La desviación típica de las pendientes simuladas coincide, salvo redondeo, con el `std err` del `summary`. Lo interesante es que statsmodels lo estima **con una sola muestra**, sin repetir el experimento, gracias a una fórmula.

### La fórmula

$$SE(\hat{\beta}_1) = \frac{s}{\sqrt{\sum_{i=1}^{n} (x_i - \bar{x})^2}}$$

- $s$ es el **error típico de la regresión**: $s = \sqrt{\sum e_i^2 / (n-2)}$, el tamaño típico de los residuos. Estima el ruido $\sigma$.
- $\sum (x_i - \bar{x})^2$ mide cuánto **varían los valores de $x$** alrededor de su media.

Como $\sum (x_i - \bar{x})^2 = (n-1)\, s_x^2$, donde $s_x$ es la desviación típica de $x$, la fórmula también se puede escribir así:

$$SE(\hat{\beta}_1) = \frac{s}{s_x \sqrt{n-1}}$$

Esta forma muestra que el error estándar depende de **tres cosas**:

1. **El ruido, $s$.** Si los puntos están lejos de la recta, la pendiente es imprecisa. Doble ruido, doble error estándar.
2. **El tamaño de la muestra, $n$.** El error estándar baja con $\sqrt{n}$: con cuatro veces más datos, la mitad de error estándar (como con la media muestral, $\sigma/\sqrt{n}$).
3. **La dispersión de $x$, $s_x$.** Con valores de $x$ muy repartidos, la pendiente se mide mejor. Con todos los $x$ amontonados, cualquier pequeño ruido inclina mucho la recta.

Calculamos el error estándar a mano con los datos de publicidad:

In [ ]:
x = publicidad["publicidad"]
n = len(publicidad)
residuos = modelo.resid

s = np.sqrt((residuos**2).sum() / (n - 2))
sxx = ((x - x.mean()) ** 2).sum()
se_formula = s / np.sqrt(sxx)

print(f"s = {s:.3f}   suma (x - media)² = {sxx:.1f}   s_x = {x.std(ddof=1):.3f}")
print(
    f"SE(b1) = {s:.3f} / raíz({sxx:.1f}) = {se_formula:.4f}   (statsmodels: {modelo.bse['publicidad']:.4f})"
)

### Los tres factores, uno a uno

Repetimos la simulación cambiando un factor cada vez: el ruido, el número de semanas y lo repartida que está la publicidad. La columna `SE simulado` es la desviación típica de las pendientes de 5 000 muestras; `SE fórmula` es $\sigma / \sqrt{\sum (x_i - \bar{x})^2}$.

In [ ]:
rng = np.random.default_rng(1)
x_base = publicidad["publicidad"].to_numpy()

escenarios = {
    "base (10 semanas, x de 8 a 20)": (x_base, 1.35),
    "doble ruido": (x_base, 2.70),
    "4 veces más semanas": (np.tile(x_base, 4), 1.35),
    "x amontonado (de 13 a 15)": (np.linspace(13, 15, 10), 1.35),
    "x muy repartido (de 2 a 26)": (np.linspace(2, 26, 10), 1.35),
}

filas = []
for nombre, (x_esc, sigma) in escenarios.items():
    pend = [
        stats.linregress(
            x_esc, beta0_real + beta1_real * x_esc + rng.normal(0, sigma, len(x_esc))
        ).slope
        for _ in range(5000)
    ]
    filas.append(
        {
            "escenario": nombre,
            "n": len(x_esc),
            "ruido σ": sigma,
            "s_x": x_esc.std(ddof=1),
            "SE simulado": np.std(pend),
            "SE fórmula": sigma / np.sqrt(((x_esc - x_esc.mean()) ** 2).sum()),
        }
    )
pd.DataFrame(filas).round(3)

Con la publicidad amontonada entre 13 y 15 mil €, el error estándar se multiplica por más de cinco, sin cambiar ni el ruido ni el número de semanas. En el gráfico siguiente, con 30 muestras simuladas en cada caso, se ve por qué: cuando los puntos están juntos (izquierda), la recta "pivota" mucho.

In [ ]:
rng = np.random.default_rng(2)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (nombre, x_esc) in zip(
    axes,
    [
        ("x amontonado (13 a 15)", np.linspace(13, 15, 10)),
        ("x repartido (2 a 26)", np.linspace(2, 26, 10)),
    ],
):
    xs = np.array([0, 28])
    for _ in range(30):
        res = stats.linregress(
            x_esc, beta0_real + beta1_real * x_esc + rng.normal(0, 1.35, len(x_esc))
        )
        ax.plot(xs, res.intercept + res.slope * xs, color="C0", alpha=0.3)
    ax.axvspan(x_esc.min(), x_esc.max(), color="C1", alpha=0.15, label="rango de x")
    ax.set_title(nombre)
    ax.set_xlabel("Publicidad (miles €)")
    ax.legend()
axes[0].set_ylabel("Ventas (miles €)")
plt.show()

**Para la empresa:** si se puede elegir cuánto invertir cada semana (un experimento), conviene probar inversiones **muy distintas**, no siempre parecidas. Se aprende más del efecto de la publicidad con semanas de 5 y de 25 mil € que con semanas de 14 y 15 mil €.

## 4. De dónde sale el p-valor

El estadístico t es la pendiente dividida por su error estándar: cuántos errores estándar se separa la pendiente estimada del 0. Si $H_0$ es cierta, sigue una distribución t de Student con $n - 2$ grados de libertad (se pierden dos por estimar $\beta_0$ y $\beta_1$). El p-valor es la probabilidad de que una t así sea, en valor absoluto, tan grande como la observada.

In [ ]:
n = len(publicidad)
b1 = modelo.params["publicidad"]
se_b1 = modelo.bse["publicidad"]

t = b1 / se_b1
p = 2 * stats.t.sf(abs(t), df=n - 2)

t_crit = stats.t.ppf(0.975, df=n - 2)
ic = (b1 - t_crit * se_b1, b1 + t_crit * se_b1)

print(f"t = {b1:.3f} / {se_b1:.3f} = {t:.2f}")
print(f"p-valor = {p:.1e}   (statsmodels: {modelo.pvalues['publicidad']:.1e})")
print(
    f"IC 95 %: [{ic[0]:.2f}, {ic[1]:.2f}]   (t crítico con {n - 2} g.l. = {t_crit:.3f})"
)

**Conclusión:** $t \approx 25$ con 8 grados de libertad, y un p-valor del orden de $10^{-9}$. Si la publicidad no tuviera relación con las ventas, sería prácticamente imposible ver una pendiente así. Cada mil euros más de publicidad se asocian con unos 3 000 € más de ventas, con un intervalo de 2.8 a 3.3 mil €.

## 5. Un caso no significativo: gasto en redes sociales

La misma tienda prueba durante diez semanas a invertir en redes sociales (miles de €) y anota las ventas.

In [ ]:
redes = pd.DataFrame(
    {
        "redes": [2.8, 3.1, 6.7, 1.6, 5.2, 6.1, 2.3, 1.4, 2.9, 5.6],
        "ventas": [56.2, 51.2, 54.0, 48.1, 56.0, 54.5, 54.0, 48.7, 52.8, 50.9],
    }
)

modelo_redes = smf.ols("ventas ~ redes", data=redes).fit()

xs = np.linspace(redes["redes"].min(), redes["redes"].max(), 50)
plt.scatter(redes["redes"], redes["ventas"])
plt.plot(
    xs, modelo_redes.params["Intercept"] + modelo_redes.params["redes"] * xs, color="C1"
)
plt.xlabel("Redes sociales (miles €)")
plt.ylabel("Ventas (miles €)")
plt.title("Redes sociales y ventas")
plt.show()

print(modelo_redes.summary().tables[1])

La pendiente estimada es positiva (unos 0.7 mil € de ventas por cada mil € en redes), pero:

- el p-valor es 0.15, mayor que 0.05;
- el intervalo de confianza va de −0.3 a 1.75: incluye el 0, y también efectos negativos y efectos de más del doble de la estimación.

¿Por qué es tan ancho? Por los factores de la sección 3: el error estándar es 0.448, casi cuatro veces el de la publicidad (0.122), porque hay el doble de ruido ($s = 2.6$ frente a 1.35) y el gasto en redes está la mitad de repartido ($s_x = 1.9$ frente a 3.7). Por eso $t = 0.72 / 0.448 = 1.6$, que no llega al valor crítico de 2.306 (t con 8 grados de libertad).

**Qué podemos concluir:** con estos datos **no podemos afirmar** que el gasto en redes esté relacionado con las ventas. Eso **no** demuestra que no lo esté: los datos son compatibles con una pendiente de 0 y también con una de 1.5. Con diez semanas y tanta variabilidad, no hay información suficiente para decidir.

De hecho, estos datos están simulados con una pendiente real de 0.8: el efecto existe, pero esta muestra no basta para detectarlo.

## 6. Un solo punto puede cambiar la conclusión

Con diez datos, cada punto pesa mucho. Un valor atípico (un error al registrar las ventas, una semana con un evento especial) puede cambiar la pendiente, su error estándar y, con ellos, la respuesta a "¿es significativa?". Veamos tres casos, añadiendo **una sola semana** a los datos:

- **A. Redes + una semana con mucho gasto y muchas ventas** (12 mil € en redes, 62 mil € de ventas).
- **B. Publicidad + una semana con ventas anómalas en el centro** (14 mil € de publicidad, 80 mil € de ventas).
- **C. Publicidad + una semana con una inversión extrema que no rindió** (30 mil € de publicidad, 60 mil € de ventas).

In [ ]:
def ajusta(df, x, y):
    m = smf.ols(f"{y} ~ {x}", data=df).fit()
    ic = m.conf_int().loc[x]
    return m, {
        "pendiente": m.params[x],
        "std err": m.bse[x],
        "p-valor": m.pvalues[x],
        "IC 2.5 %": ic[0],
        "IC 97.5 %": ic[1],
        "R²": m.rsquared,
    }


casos = [
    ("A. Redes", redes, "redes", pd.DataFrame({"redes": [12], "ventas": [62]})),
    (
        "B. Publicidad, atípico en y",
        publicidad,
        "publicidad",
        pd.DataFrame({"publicidad": [14], "ventas": [80]}),
    ),
    (
        "C. Publicidad, x extremo",
        publicidad,
        "publicidad",
        pd.DataFrame({"publicidad": [30], "ventas": [60]}),
    ),
]

filas = []
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (nombre, df, x, extra) in zip(axes, casos):
    df_extra = pd.concat([df, extra], ignore_index=True)
    m_sin, r_sin = ajusta(df, x, "ventas")
    m_con, r_con = ajusta(df_extra, x, "ventas")
    filas.append({"caso": nombre, "datos": "sin el punto", **r_sin})
    filas.append({"caso": nombre, "datos": "con el punto", **r_con})

    xs = np.linspace(0, df_extra[x].max() * 1.05, 50)
    ax.scatter(df[x], df["ventas"], color="C0")
    ax.scatter(
        extra[x], extra["ventas"], color="C3", s=80, marker="D", label="punto añadido"
    )
    ax.plot(
        xs,
        m_sin.params["Intercept"] + m_sin.params[x] * xs,
        color="C0",
        label="recta sin el punto",
    )
    ax.plot(
        xs,
        m_con.params["Intercept"] + m_con.params[x] * xs,
        color="C3",
        ls="--",
        label="recta con el punto",
    )
    ax.set_xlim(0, xs.max())
    ax.set_title(nombre)
    ax.set_xlabel(f"{x} (miles €)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Ventas (miles €)")
plt.show()

pd.DataFrame(filas).set_index(["caso", "datos"]).round(3)

Cada caso enseña algo distinto:

- **A.** Un punto lejos del resto en $x$ y en la dirección de la tendencia **convierte una pendiente no significativa en significativa** (el p-valor pasa de 0.15 a 0.004). La "relación" se apoya casi entera en una semana.
- **B.** Un atípico en $y$ con $x$ en el centro apenas mueve la pendiente, pero **dispara el ruido** $s$, y con él el error estándar (de 0.12 a 0.67): el intervalo pasa de [2.8, 3.3] a [1.6, 4.6] y $R^2$ baja de 0.99 a 0.70.
- **C.** Un punto con un $x$ muy alejado tiene mucho **apalancamiento**: tira de la recta hacia él, y la pendiente baja de 3.05 a 1.19.

La regla que se deduce es que **el p-valor no avisa** de que depende de un solo punto. Por eso, antes de interpretar el `summary`:

1. Dibuja los datos con la recta.
2. Si un punto se separa del resto, averigua por qué: puede ser un error de registro o la semana más informativa de todas.
3. Ajusta el modelo con y sin ese punto. Si la conclusión cambia, dilo en el informe: la evidencia es débil.

No se borra un punto solo porque estropea el resultado.

## 7. Simulación: ¿qué pasa si la pendiente real es 0?

Generamos 5 000 muestras de 10 semanas con el mismo gasto en redes que nuestros datos y el mismo ruido ($\sigma = 2.6$), pero en las que el gasto en redes **no** influye en las ventas ($\beta_1 = 0$), y ajustamos una recta a cada una. Las pendientes estimadas no son 0: varían por azar.

In [ ]:
rng = np.random.default_rng(42)
x_redes = redes["redes"].to_numpy()
t_redes = modelo_redes.tvalues["redes"]

pendientes, tvalores, pvalores = [], [], []
for _ in range(5000):
    y_sim = 50 + 0 * x_redes + rng.normal(0, 2.6, len(x_redes))  # pendiente real = 0
    res = stats.linregress(x_redes, y_sim)
    pendientes.append(res.slope)
    tvalores.append(res.slope / res.stderr)
    pvalores.append(res.pvalue)

pendientes, tvalores, pvalores = (
    np.array(pendientes),
    np.array(tvalores),
    np.array(pvalores),
)

plt.hist(pendientes, bins=50)
plt.axvline(
    modelo_redes.params["redes"],
    color="C1",
    ls="--",
    label="pendiente de los datos de redes",
)
plt.xlabel("Pendiente estimada")
plt.ylabel("Número de muestras")
plt.title("Pendientes estimadas cuando la pendiente real es 0")
plt.legend()
plt.show()

print(
    f"Muestras con |t| >= {t_redes:.2f} (el t de los datos de redes): {(np.abs(tvalores) >= t_redes).mean():.1%}"
)
print(f"p-valor de los datos de redes: {modelo_redes.pvalues['redes']:.1%}")
print(f"Muestras con p < 0.05 (falsos positivos): {(pvalores < 0.05).mean():.1%}")

Dos lecturas:

- Alrededor del 15 % de las muestras sin ninguna relación dan un $t$ tan extremo como el de los datos de redes. **Eso es exactamente el p-valor**: la proporción de muestras que, si $H_0$ fuera cierta, darían un resultado tan extremo como el observado. Por eso la pendiente de redes no es convincente: el azar la explica bien.
- Alrededor del 5 % de las muestras dan p < 0.05 aunque no haya relación. Es el **error de tipo I**: con un umbral de 0.05, una de cada veinte veces "encontraremos" una relación que no existe.

## 8. El tamaño de la muestra importa

Ahora la pendiente real es 0.8, como en los datos de redes, con el mismo ruido y un gasto en redes repartido entre 1 y 8 mil €. Para cada tamaño de muestra generamos 1 000 muestras y contamos en cuántas sale la pendiente significativa (es la **potencia** del contraste), y cuánto mide de media el intervalo de confianza.

In [ ]:
rng = np.random.default_rng(7)
filas = []
for n in [10, 20, 40, 100, 200]:
    pvals, anchos = [], []
    for _ in range(1000):
        x = rng.uniform(1, 8, n)
        y = 50 + 0.8 * x + rng.normal(0, 2.6, n)  # pendiente real = 0.8
        res = stats.linregress(x, y)
        t_crit = stats.t.ppf(0.975, df=n - 2)
        pvals.append(res.pvalue)
        anchos.append(2 * t_crit * res.stderr)
    filas.append(
        {
            "n": n,
            "% muestras con p < 0.05": 100 * np.mean(np.array(pvals) < 0.05),
            "ancho medio del IC": np.mean(anchos),
        }
    )
pd.DataFrame(filas).round(2)

Con 10 semanas, el efecto real solo se detecta en un tercio de las muestras: lo normal es que salga "no significativo", como en los datos de redes. Con 40 semanas ya se detecta casi siempre, y el intervalo es la mitad de ancho que con 10 (el error estándar baja con $\sqrt{n}$, ver la sección 3).

"No significativo" depende de cuántos datos tenemos: un efecto real pero pequeño necesita muestras grandes para detectarse. Al revés, con muchísimos datos, pendientes muy pequeñas y sin importancia práctica salen significativas. Por eso hay que mirar siempre el **intervalo de confianza** y preguntarse si el tamaño del efecto importa para la decisión, no solo el p-valor.

## 9. Cinco formas de ajustar la misma recta

En Python hay al menos cinco funciones que ajustan una recta por mínimos cuadrados. Todas dan los **mismos coeficientes**; cambia cómo se llaman y qué más devuelven:

| Función | Librería | Para qué está pensada |
|---|---|---|
| `np.polyfit(x, y, 1)` | NumPy | ajustar un polinomio, sin estadística |
| `stats.linregress(x, y)` | SciPy | regresión **simple** rápida, con p-valor de la pendiente |
| `smf.ols("y ~ x", data=df)` | statsmodels (fórmulas) | inferencia completa, escribiendo el modelo como fórmula |
| `sm.OLS(y, X)` | statsmodels (matrices) | lo mismo, pasando las matrices |
| `LinearRegression().fit(X, y)` | scikit-learn | predecir, dentro de un flujo de *machine learning* |

In [ ]:
x = publicidad["publicidad"]  # Serie, 1 dimensión
X = publicidad[["publicidad"]]  # DataFrame, 2 dimensiones (doble corchete)
y = publicidad["ventas"]

# 1. NumPy: devuelve [pendiente, intercepto], en ese orden
pendiente_np, intercepto_np = np.polyfit(x, y, 1)

# 2. SciPy: solo admite una variable x
lr = stats.linregress(x, y)

# 3. statsmodels con fórmula: añade el intercepto solo
m_formula = smf.ols("ventas ~ publicidad", data=publicidad).fit()

# 4. statsmodels con matrices: el intercepto hay que añadirlo con add_constant
m_matrices = sm.OLS(y, sm.add_constant(X)).fit()

# 5. scikit-learn: X en 2 dimensiones; ojo al orden (X, y)
sk = LinearRegression().fit(X, y)

comparacion = pd.DataFrame(
    {
        "np.polyfit": [intercepto_np, pendiente_np, np.nan, np.nan, np.nan],
        "linregress": [lr.intercept, lr.slope, lr.rvalue**2, lr.stderr, lr.pvalue],
        "smf.ols": [
            m_formula.params["Intercept"],
            m_formula.params["publicidad"],
            m_formula.rsquared,
            m_formula.bse["publicidad"],
            m_formula.pvalues["publicidad"],
        ],
        "sm.OLS": [
            m_matrices.params["const"],
            m_matrices.params["publicidad"],
            m_matrices.rsquared,
            m_matrices.bse["publicidad"],
            m_matrices.pvalues["publicidad"],
        ],
        "LinearRegression": [
            sk.intercept_,
            sk.coef_[0],
            sk.score(X, y),
            np.nan,
            np.nan,
        ],
    },
    index=["b0", "b1", "R²", "SE(b1)", "p-valor b1"],
)
comparacion.T

Los coeficientes y el $R^2$ coinciden en todas, porque todas resuelven el mismo problema de mínimos cuadrados. Las casillas vacías (`NaN`) son lo que cada función **no** calcula: `np.polyfit` y `LinearRegression` no dan errores estándar ni p-valores (y `np.polyfit` tampoco da $R^2$). Para saber si la pendiente es significativa sirven `linregress` (solo en regresión simple) y statsmodels.

Detalles que suelen dar problemas:

- `linregress` devuelve `rvalue`, que es la **correlación** $r$, no $R^2$: hay que elevarla al cuadrado. Su `pvalue` es el de la pendiente, y `intercept_stderr` el error estándar del intercepto.
- `np.polyfit` devuelve los coeficientes de mayor a menor grado: primero la pendiente y después el intercepto.
- `sm.OLS` recibe `(y, X)` y `LinearRegression().fit` recibe `(X, y)`: el orden está cambiado.
- `LinearRegression` necesita `X` en dos dimensiones (`df[["x"]]`), aunque haya una sola variable; `linregress` y `polyfit` la quieren en una (`df["x"]`); `sm.OLS` acepta las dos.
- `smf.ols` y `sm.OLS` son el mismo modelo con la misma salida: la fórmula añade el intercepto y crea las variables dummy de las categóricas por nosotros (`C(variable)`).

Las predicciones también coinciden:

In [ ]:
nuevas = pd.DataFrame({"publicidad": [9, 17]})
print("LinearRegression:", sk.predict(nuevas).round(2))
print("smf.ols:         ", m_formula.predict(nuevas).round(2).values)
print(
    "linregress:      ",
    (lr.intercept + lr.slope * nuevas["publicidad"]).round(2).values,
)

statsmodels, además, da el intervalo de cada predicción: `mean_ci` para la venta **media** de las semanas con esa publicidad y `obs_ci` para la venta de **una** semana concreta, que es más ancho.

In [ ]:
m_formula.get_prediction(nuevas).summary_frame(alpha=0.05).round(2)

### Cuidado con `sm.OLS` sin constante

Si se olvida `sm.add_constant`, `sm.OLS` ajusta una recta que pasa por el origen, con otros coeficientes y otro $R^2$.

In [ ]:
sin_constante = sm.OLS(y, X).fit()
con_constante = sm.OLS(y, sm.add_constant(X)).fit()

print(
    "Sin constante:",
    sin_constante.params.round(3).to_dict(),
    f"R² = {sin_constante.rsquared:.4f}",
)
print(
    "Con constante:",
    con_constante.params.round(3).to_dict(),
    f"R² = {con_constante.rsquared:.4f}",
)

El $R^2$ sin constante parece incluso mejor, pero no es comparable: statsmodels lo calcula respecto a 0 en lugar de respecto a la media de $Y$. El summary lo avisa con `R-squared (uncentered)`.

## 10. Resumen: ¿qué uso?

| | `np.polyfit` | `stats.linregress` | `smf.ols` / `sm.OLS` | `LinearRegression` |
|---|---|---|---|---|
| Variables explicativas | una (o potencias de una) | **solo una** | varias | varias |
| Intercepto | sí | sí | fórmula: automático; `sm.OLS`: `add_constant` | automático (`fit_intercept=True`) |
| Formato de `x` | 1 dimensión | 1 dimensión | fórmula + DataFrame; `sm.OLS`: 1 o 2 dimensiones | 2 dimensiones |
| Coeficientes | `[b1, b0]` | `slope`, `intercept` | `params` | `coef_`, `intercept_` |
| $R^2$ | no | `rvalue**2` | `rsquared`, `rsquared_adj` | `score(X, y)` |
| Errores estándar, p-valores | no | SE de pendiente e intercepto; p-valor solo de la pendiente | todos (`summary()`, `bse`, `pvalues`, `conf_int()`) | no |
| Intervalos de confianza | no | a mano con `stderr` | `conf_int()` | no |
| Intervalos de predicción | no | no | `get_prediction().summary_frame()` | no |
| Train/test, validación cruzada, pipelines | no | no | a mano | sí |
| Cuándo usarla | dibujar una recta de tendencia | contraste rápido en regresión simple | **explicar**: ¿qué variables importan y cuánto? | **predecir**: evaluar con datos nuevos |

## 11. Para pensar

1. En los datos de redes, ¿qué le dirías al director de marketing que quiere dejar de invertir en redes "porque no tiene efecto"?
2. En la sección 8, prueba otros valores de `n`. ¿Cuántas semanas hacen falta para detectar una pendiente real de 0.8 en al menos el 80 % de las muestras?
3. En la simulación de la sección 7, cambia el umbral a 0.01. ¿Qué proporción de muestras da un falso positivo?
4. En el caso A de la sección 6, cambia el punto añadido a 12 mil € en redes y 50 mil € de ventas. ¿Qué pasa con la pendiente y su p-valor?
5. Ajusta la recta de los datos de redes con `linregress` y con `LinearRegression`. ¿Con cuál puedes saber si la pendiente es significativa?